In [2]:
# Pkg.add("Graphs")
import Pkg; Pkg.add("GraphPlot")
using LightGraphs
using Random
Random.seed!(123) #123:30; 1234:40; 12345:50
# using Graphs
# using Pkg
# Pkg.add("GraphPlot")
using GraphPlot

### Get y_0 ###
function gx_bound(c_L, c_U, c, c_g, x_now, edge)
    
    #println(f,"Current CELL's LB = ", c_L)
    #println(f,"Current CELL's UB = ", c_U)
    #println(f, "Current interdiction x = ", x_now)
    Len = length(edge[:,1])
    start_node = edge[:,1]
    end_node = edge[:,2]

    no_node = max(maximum(start_node), maximum(end_node) )
    no_link = length(start_node)


    function getShortestX(state, start_node, end_node, origin, destination)
        _x = zeros(Int, length(start_node))
        _path = enumerate_paths(state, destination)

        for i=1:length(_path)-1
            _start = _path[i]
            _end = _path[i+1]

            for j=1:length(start_node)
                if start_node[j]==_start && end_node[j]==_end
                _x[j] = 1
                break
                end
            end

        end
        _x
    end


    graph = Graph(no_node)
    distmx = Inf*ones(no_node, no_node)

    # Adding links to the graph
    for i=1:no_link
        add_edge!(graph, start_node[i], end_node[i])
        distmx[start_node[i], end_node[i]] = c_g[i]
    end

    # Run Dijkstra's Algorithm from the origin node to all nodes
    state = dijkstra_shortest_paths(graph, origin, distmx)
    label = state.dists
    pred = state.parents
    b_arc = ""
    
    for i = 1: length(state.parents)
        if state.parents[i] != 0 
            b_arc = string(b_arc, "(", state.parents[i], ",", i, ")")
        end
    end
    
    # Retrieving the shortest path
    path = enumerate_paths(state, destination)
    
    #parents = LightGraphs.DijkstraState(state, destination)

    # Retrieving the 'x' variable in a 0-1 vector
    y = getShortestX(state, start_node, end_node, origin, destination)
    #println(f,"y vector:", y)
    
    gx = sum(c_g[i]*y[i] for i = 1:no_link)    
    SP = sum(c[i]*y[i] for i = 1:length(c))
    T = Int64[]
    for i = 1:Len
        if pred[edge[i,2]] == edge[i,1]
            push!(T, 1)
        else
            push!(T, 0)
        end
    end

    y_index = findall(y .== 1)

#     println("Minimum Spanning Tree: ", T)
    #println("Shortest path y = ", y)
    #println("Indices of shortest path (edge) = ", y_index)
    #println("Nodes visited =", path)
    #println("Minimum Spanning Tree =", pred)
    #println("Node label =" ,label)

    return y, gx, SP
end
######

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.9/Project.toml`
  No Changes to `~/.julia/environments/v1.9/Manifest.toml`


gx_bound (generic function with 1 method)

In [4]:
#Orig 
using JuMP
N = 30
target_density = 15
A = (target_density/100)*(N)*(N-1)
max_A = N*(N-1)
density = (target_density/100)*max_A/(max_A - (N-1) - (N-2))
println("density ", density)
println("max_A ",max_A)
numInstances = 800
all_Density = zeros(numInstances)
unc_arc = 0.15
myInstance = 1

origin = 1
destination = N

while myInstance <= numInstances
    try
        print(myInstance)
        edge = Array{Int64}(undef,(0,2))
        for i = 1:N-1
            for j = 2:N
                r = rand()
                if r <= density && i!=j
                    arc = [i,j]
                    edge = [edge; [i,j]']
                end
            end
        end
    #     avg = avg + length(edge[:,1])/max_A
        cU_orig = zeros(length(edge[:,1]))
        cL_orig = zeros(length(edge[:,1]))
        d = zeros(length(edge[:,1]))
        origin = 1
        destination = N
    #     println("",edge)
        Len = length(edge[:,1])
    #     println("Len = ", length(edge[:,1]))
        for i = 1:length(edge[:,1])
             
    #         prob = rand(1:10)
    #         high = rand(1:40)
    #         if prob <= 3
    #             low = rand(1:high)
    #         else
    #             low = high
    #         end
            diff = abs(edge[i,2]-edge[i,1])*20
            c = rand(1:diff)
            r = rand()
    #         println("c = ", c)
    #         println("cL_orig ", cL_orig)
            if r <= unc_arc
                unc_amt = rand(1:c)
                cL_orig[i] = c - unc_amt
                cU_orig[i] = c + unc_amt
            else
                cL_orig[i] = c 
                cU_orig[i] = c 
            end
            
            interdict = rand(1:50)
    #         push!(cU_orig, high)
    #         push!(cL_orig, low)
            d[i] = interdict
        end
        c_L = cL_orig
        c_U = cU_orig
        c = (c_L + c_U)/2
        c_g = c 
        x_now = zeros(Int64,length(d))
    #     println("1")
        y,gx,SP = gx_bound(c_L, c_U, c, c_g, x_now, edge)
        
        
        println("edge = ", edge[y.>0,:])
    #     println(y)
        
        if sum(y[i] for i = 1:length(d)) > 0  
            println(edge[y.>0,:])
    #         outfile = "./PrelimInstances/N"*string(N)*"_d"*string(target_density)*"_Ins_"*string(myInstance)*".jl"
    #         f = open(outfile, "w")
    #         println(f,"edge = ", edge)
    #         println(f,"cL_orig = ", cL_orig)
    #         println(f,"cU_orig = ", cU_orig)
    #         println(f,"d = ", d)
    #         Len = length(d)
    #         #STARTING SOLUTION:
    #         println(f, "Len = length(d)
    #         \nyy = ",y, "
    #         \nc_orig = 0.5*(cL_orig+cU_orig)
    #         \nSP_init = sum(yy[i]*c_orig[i] for i = 1:Len)
    
    #         \np = [1.0]
    #         \ng = [SP_init]
    #         \nh = [0.0]
    
    #         \norigin = ",1,"
    #         \ndestination =",N,"
    
    #         last_node = maximum(edge)
    #         all_nodes = collect(1:last_node)
    
    #         M_orig = zeros(Len)
    
    #         for i = 1:Len
    #             M_orig[i] = cU_orig[i] - cL_orig[i]
    #         end
    
    #         case = 0
    #         delta1 = 1e-6
    #         delta2 = ",2,"
    #         last_node = maximum(edge)")
    #         close(f)
            # f = open("./NewCSVFeb24/N"*string(N)*"_"*string(myInstance)*".csv", "w")
            if myInstance > 500
                f = open("./PythonConversion/NewCSVDec2024/"*string(target_density)*"_N"*string(N)*"_"*string(myInstance)*".csv", "w")
                print("Here1")
                write(f, string(Len),"\n")
                # println(cL_orig[1])
                write(f,string(origin-1),"\n")
                write(f,string(destination-1),"\n")
                
                write(f,"\n")
                print("Here")
                for e =1:Len
                    write(f, string(edge[e, 1]-1), "\t", string(edge[e, 2]-1),"\t", string(cL_orig[e]),"\t", string(cU_orig[e]),"\t", string(d[e]),"\n")
                end
        
                close(f)
            end
            myInstance = myInstance + 1
        end
        
    catch
        @warn No file generated
    end
end

# println(avg/numInstances)

density 0.16051660516605165
max_A 870
1edge = [1 8; 3 27; 8 21; 21 3; 27 30]
[1 8; 3 27; 8 21; 21 3; 27 30]
2edge = [1 20; 18 19; 19 27; 20 18; 27 30]
[1 20; 18 19; 19 27; 20 18; 27 30]
3edge = [1 8; 4 10; 8 4; 10 12; 12 13; 13 26; 24 27; 26 24; 27 30]
[1 8; 4 10; 8 4; 10 12; 12 13; 13 26; 24 27; 26 24; 27 30]
4edge = [1 3; 3 22; 18 21; 21 23; 22 18; 23 24; 24 30]
[1 3; 3 22; 18 21; 21 23; 22 18; 23 24; 24 30]
5edge = [1 9; 9 16; 16 22; 22 23; 23 24; 24 29; 29 30]
[1 9; 9 16; 16 22; 22 23; 23 24; 24 29; 29 30]
6edge = [1 22; 22 25; 25 30]
[1 22; 22 25; 25 30]
7edge = [1 5; 5 30]
[1 5; 5 30]
8edge = [1 3; 3 13; 13 30]
[1 3; 3 13; 13 30]
9edge = [1 2; 2 26; 26 30]
[1 2; 2 26; 26 30]
10edge = [1 3; 3 11; 8 30; 11 8]
[1 3; 3 11; 8 30; 11 8]
11edge = [1 17; 5 30; 6 11; 11 5; 17 6]
[1 17; 5 30; 6 11; 11 5; 17 6]
12edge = [1 18; 3 30; 9 10; 10 3; 11 9; 18 23; 19 11; 23 19]
[1 18; 3 30; 9 10; 10 3; 11 9; 18 23; 19 11; 23 19]
13edge = [1 27; 24 30; 27 24]
[1 27; 24 30; 27 24]
14edge = [1 4; 4 1

┌ Error: Exception while generating log record in module Main at In[4]:137
│   exception =
│    UndefVarError: `No` not defined
│    Stacktrace:
│      [1] backtrace()
│        @ Base ./error.jl:114
│      [2] logging_error(logger::Any, level::Any, _module::Any, group::Any, id::Any, filepath::Any, line::Any, err::Any, real::Bool)
│        @ Base.CoreLogging ./logging.jl:465
│      [3] invokelatest(::Any, ::Any, ::Vararg{Any}; kwargs::Base.Pairs{Symbol, Union{}, Tuple{}, NamedTuple{(), Tuple{}}})
│        @ Base ./essentials.jl:816
│      [4] invokelatest(::Any, ::Any, ::Vararg{Any})
│        @ Base ./essentials.jl:813
│      [5] macro expansion
│        @ logging.jl:352 [inlined]
│      [6] top-level scope
│        @ In[4]:137
│      [7] eval
│        @ ./boot.jl:370 [inlined]
│      [8] include_string(mapexpr::typeof(REPL.softscope), mod::Module, code::String, filename::String)
│        @ Base ./loading.jl:1864
│      [9] softscope_include_string(m::Module, code::String, filename::Str

[1 12; 12 18; 18 25; 25 26; 26 30]
[1 12; 12 18; 18 25; 25 26; 26 30]
229edge = [1 16; 15 30; 16 18; 18 15]
[1 16; 15 30; 16 18; 18 15]
230edge = [1 15; 15 19; 19 26; 26 30]
[1 15; 15 19; 19 26; 26 30]
231edge = [1 10; 10 16; 16 23; 23 27; 27 30]
[1 10; 10 16; 16 23; 23 27; 27 30]
232edge = [1 4; 4 30]
[1 4; 4 30]
233edge = [1 5; 5 11; 11 30]
[1 5; 5 11; 11 30]
234edge = [1 15; 13 30; 15 16; 16 28; 27 13; 28 27]
[1 15; 13 30; 15 16; 16 28; 27 13; 28 27]
235edge = [1 13; 13 21; 19 30; 21 19]
[1 13; 13 21; 19 30; 21 19]
236edge = [1 4; 4 16; 14 30; 15 17; 16 15; 17 14]
[1 4; 4 16; 14 30; 15 17; 16 15; 17 14]
237edge = [1 15; 8 30; 15 8]
[1 15; 8 30; 15 8]
238edge = [1 8; 8 20; 20 22; 22 29; 29 30]
[1 8; 8 20; 20 22; 22 29; 29 30]
239edge = [1 26; 18 30; 22 23; 23 18; 26 22]
[1 26; 18 30; 22 23; 23 18; 26 22]
240edge = [1 24; 24 29; 26 30; 29 26]
[1 24; 24 29; 26 30; 29 26]
241edge = [1 10; 10 14; 11 27; 12 11; 14 12; 27 30]
[1 10; 10 14; 11 27; 12 11; 14 12; 27 30]
242edge = [1 2; 2 8; 8

┌ Error: Exception while generating log record in module Main at In[4]:137
│   exception =
│    UndefVarError: `No` not defined
│    Stacktrace:
│      [1] logging_error(logger::Any, level::Any, _module::Any, group::Any, id::Any, filepath::Any, line::Any, err::Any, real::Bool)
│        @ Base.CoreLogging ./logging.jl:465
│      [2] invokelatest(::Any, ::Any, ::Vararg{Any}; kwargs::Base.Pairs{Symbol, Union{}, Tuple{}, NamedTuple{(), Tuple{}}})
│        @ Base ./essentials.jl:816
│      [3] invokelatest(::Any, ::Any, ::Vararg{Any})
│        @ Base ./essentials.jl:813
│      [4] macro expansion
│        @ logging.jl:352 [inlined]
│      [5] top-level scope
│        @ In[4]:137
│      [6] eval
│        @ ./boot.jl:370 [inlined]
│      [7] include_string(mapexpr::typeof(REPL.softscope), mod::Module, code::String, filename::String)
│        @ Base ./loading.jl:1864
│      [8] softscope_include_string(m::Module, code::String, filename::String)
│        @ SoftGlobalScope ~/.julia/packages/Soft

[1 5; 5 7; 7 25; 25 29; 29 30]
[1 5; 5 7; 7 25; 25 29; 29 30]
335edge = [1 8; 8 11; 11 27; 24 30; 27 24]
[1 8; 8 11; 11 27; 24 30; 27 24]
336edge = [1 5; 2 20; 5 2; 20 30]
[1 5; 2 20; 5 2; 20 30]
337edge = [1 8; 8 30]
[1 8; 8 30]
338edge = [1 18; 18 25; 22 30; 25 22]
[1 18; 18 25; 22 30; 25 22]
339edge = [1 9; 9 19; 19 24; 24 30]
[1 9; 9 19; 19 24; 24 30]
340edge = [1 27; 27 30]
[1 27; 27 30]
341edge = [1 14; 14 16; 16 30]
[1 14; 14 16; 16 30]
342edge = [1 12; 12 18; 18 30]
[1 12; 12 18; 18 30]
343edge = [1 25; 24 27; 25 24; 26 30; 27 26]
[1 25; 24 27; 25 24; 26 30; 27 26]
344edge = [1 6; 6 16; 12 30; 16 12]
[1 6; 6 16; 12 30; 16 12]
345edge = [1 2; 2 3; 3 22; 19 30; 22 19]
[1 2; 2 3; 3 22; 19 30; 22 19]
346edge = [1 5; 5 26; 16 30; 26 16]
[1 5; 5 26; 16 30; 26 16]
347edge = [1 10; 10 11; 11 30]
[1 10; 10 11; 11 30]
348edge = [1 3; 3 6; 6 14; 12 18; 14 12; 18 30]
[1 3; 3 6; 6 14; 12 18; 14 12; 18 30]
349edge = [1 8; 8 10; 10 11; 11 18; 18 19; 19 30]
[1 8; 8 10; 10 11; 11 18; 18 19; 19 

...

[1 10; 8 13; 10 8; 13 30]
[1 10; 8 13; 10 8; 13 30]
Here1Here564edge = [1 13; 13 27; 27 30]
[1 13; 13 27; 27 30]
Here1Here565edge = [1 6; 6 21; 21 27; 27 30]
[1 6; 6 21; 21 27; 27 30]
Here1Here566edge = [1 28; 21 29; 22 23; 23 21; 28 22; 29 30]
[1 28; 21 29; 22 23; 23 21; 28 22; 29 30]
Here1Here567edge = [1 3; 3 5; 5 26; 26 27; 27 30]
[1 3; 3 5; 5 26; 26 27; 27 30]
Here1Here568edge = [1 29; 29 30]
[1 29; 29 30]
Here1Here569edge = [1 8; 8 18; 18 30]
[1 8; 8 18; 18 30]
Here1Here570edge = [1 4; 4 12; 12 15; 15 30]
[1 4; 4 12; 12 15; 15 30]
Here1Here571edge = [1 27; 12 30; 18 12; 19 18; 27 19]
[1 27; 12 30; 18 12; 19 18; 27 19]
Here1Here572edge = [1 7; 4 6; 6 25; 7 18; 18 4; 25 30]
[1 7; 4 6; 6 25; 7 18; 18 4; 25 30]
Here1Here573edge = [1 27; 17 21; 21 30; 22 17; 26 22; 27 26]
[1 27; 17 21; 21 30; 22 17; 26 22; 27 26]
Here1Here574edge = [1 11; 11 14; 14 23; 23 30]
[1 11; 11 14; 14 23; 23 30]
Here1Here575edge = [1 4; 4 6; 6 10; 10 26; 26 30]
[1 4; 4 6; 6 10; 10 26; 26 30]
Here1Here576edge =

┌ Error: Exception while generating log record in module Main at In[4]:137
│   exception =
│    UndefVarError: `No` not defined
│    Stacktrace:
│      [1] logging_error(logger::Any, level::Any, _module::Any, group::Any, id::Any, filepath::Any, line::Any, err::Any, real::Bool)
│        @ Base.CoreLogging ./logging.jl:465
│      [2] invokelatest(::Any, ::Any, ::Vararg{Any}; kwargs::Base.Pairs{Symbol, Union{}, Tuple{}, NamedTuple{(), Tuple{}}})
│        @ Base ./essentials.jl:816
│      [3] invokelatest(::Any, ::Any, ::Vararg{Any})
│        @ Base ./essentials.jl:813
│      [4] macro expansion
│        @ logging.jl:352 [inlined]
│      [5] top-level scope
│        @ In[4]:137
│      [6] eval
│        @ ./boot.jl:370 [inlined]
│      [7] include_string(mapexpr::typeof(REPL.softscope), mod::Module, code::String, filename::String)
│        @ Base ./loading.jl:1864
│      [8] softscope_include_string(m::Module, code::String, filename::String)
│        @ SoftGlobalScope ~/.julia/packages/Soft

694edge = [1 9; 9 30]
[1 9; 9 30]
Here1Here695edge = [1 2; 2 3; 3 6; 6 7; 7 30]
[1 2; 2 3; 3 6; 6 7; 7 30]
Here1Here696edge = [1 2; 2 7; 7 22; 22 26; 26 30]
[1 2; 2 7; 7 22; 22 26; 26 30]
Here1Here697edge = [1 5; 5 30]
[1 5; 5 30]
Here1Here698edge = [1 3; 3 24; 24 30]
[1 3; 3 24; 24 30]
Here1Here699edge = [1 18; 18 21; 21 30]
[1 18; 18 21; 21 30]
Here1Here700edge = [1 30]
[1 30]
Here1Here701edge = [1 3; 3 11; 11 21; 21 30]
[1 3; 3 11; 11 21; 21 30]
Here1Here702edge = [1 25; 9 20; 20 29; 25 9; 29 30]
[1 25; 9 20; 20 29; 25 9; 29 30]
Here1Here703edge = [1 22; 22 27; 27 30]
[1 22; 22 27; 27 30]
Here1Here704edge = [1 16; 16 29; 29 30]
[1 16; 16 29; 29 30]
Here1Here705edge = [1 15; 8 30; 9 8; 12 9; 13 12; 15 16; 16 13]
[1 15; 8 30; 9 8; 12 9; 13 12; 15 16; 16 13]
Here1Here706edge = [1 18; 18 28; 28 30]
[1 18; 18 28; 28 30]
Here1Here707edge = [1 4; 4 30]
[1 4; 4 30]
Here1Here708edge = [1 14; 14 23; 19 30; 22 19; 23 22]
[1 14; 14 23; 19 30; 22 19; 23 22]
Here1Here709edge = [1 15; 9 11; 11 27;

┌ Error: Exception while generating log record in module Main at In[4]:137
│   exception =
│    UndefVarError: `No` not defined
│    Stacktrace:
│      [1] logging_error(logger::Any, level::Any, _module::Any, group::Any, id::Any, filepath::Any, line::Any, err::Any, real::Bool)
│        @ Base.CoreLogging ./logging.jl:465
│      [2] invokelatest(::Any, ::Any, ::Vararg{Any}; kwargs::Base.Pairs{Symbol, Union{}, Tuple{}, NamedTuple{(), Tuple{}}})
│        @ Base ./essentials.jl:816
│      [3] invokelatest(::Any, ::Any, ::Vararg{Any})
│        @ Base ./essentials.jl:813
│      [4] macro expansion
│        @ logging.jl:352 [inlined]
│      [5] top-level scope
│        @ In[4]:137
│      [6] eval
│        @ ./boot.jl:370 [inlined]
│      [7] include_string(mapexpr::typeof(REPL.softscope), mod::Module, code::String, filename::String)
│        @ Base ./loading.jl:1864
│      [8] softscope_include_string(m::Module, code::String, filename::String)
│        @ SoftGlobalScope ~/.julia/packages/Soft

[1 7; 7 8; 8 26; 26 30]
[1 7; 7 8; 8 26; 26 30]
Here1Here745edge = [1 8; 8 27; 23 30; 27 23]
[1 8; 8 27; 23 30; 27 23]
Here1Here746edge = [1 6; 6 30]
[1 6; 6 30]
Here1Here747edge = [1 3; 3 12; 12 21; 18 19; 19 30; 21 18]
[1 3; 3 12; 12 21; 18 19; 19 30; 21 18]
Here1Here748edge = [1 9; 9 12; 12 26; 26 29; 29 30]
[1 9; 9 12; 12 26; 26 29; 29 30]
Here1Here749edge = [1 2; 2 19; 19 30]
[1 2; 2 19; 19 30]
Here1Here750edge = [1 9; 9 10; 10 11; 11 20; 15 27; 19 15; 20 19; 26 30; 27 26]
[1 9; 9 10; 10 11; 11 20; 15 27; 19 15; 20 19; 26 30; 27 26]
Here1Here751edge = [1 15; 15 19; 19 30]
[1 15; 15 19; 19 30]
Here1Here752edge = [1 9; 9 13; 13 30]
[1 9; 9 13; 13 30]
Here1Here753edge = [1 6; 6 12; 10 14; 12 10; 14 25; 25 30]
[1 6; 6 12; 10 14; 12 10; 14 25; 25 30]
Here1Here754edge = [1 3; 3 9; 5 22; 9 5; 22 30]
[1 3; 3 9; 5 22; 9 5; 22 30]
Here1Here755edge = [1 25; 17 30; 19 17; 24 19; 25 24]
[1 25; 17 30; 19 17; 24 19; 25 24]
Here1Here756edge = [1 12; 12 27; 21 30; 27 21]
[1 12; 12 27; 21 30; 27 21

In [26]:
for i = 1:10
    include("./NewCSVDec2024/20_N40_"*string(i)*".csv")
    println("M = ", M_orig[M_orig.>0])
end

LoadError: SystemError: opening file "/Users/dinguyen/Desktop/Local Documents/GitHub/Paper5/NewCSVDec2024/20_N40_1.csv": No such file or directory

In [ ]:
rand(1:999)/1000